# Chạy LLM extractor (Qwen2.5, few-shot) trên Google Colab
Self-host ≤9B, KHÔNG API ngoài. Runtime → Change runtime type → **GPU** (T4/L4/A100).
Model lấy từ `configs/config.yaml → extract.llm_model` (mặc định **3B**; đổi **7B** nếu muốn few-shot mạnh hơn — cell 5 có sẵn dòng đổi).
Mẹo: cache model vào Google Drive (cell 1) để khỏi tải lại 15GB mỗi phiên.

In [1]:
# 1) (Khuyến nghị) Mount Drive + cache HuggingFace vào Drive để khỏi tải lại model
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    print('HF_HOME =', os.environ['HF_HOME'])
except Exception as e:
    print('Bỏ qua Drive:', e)

Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/hf_cache


In [2]:
# 2) Lấy code
%cd /content
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /content/viettel-medreason
!git pull -q

/content
Cloning into 'viettel-medreason'...
remote: Enumerating objects: 756, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 756 (delta 38), reused 40 (delta 28), pack-reused 667 (from 1)
Receiving objects: 100% (756/756), 4.38 MiB | 16.02 MiB/s, done.
Resolving deltas: 100% (351/351), done.
/content/viettel-medreason


In [3]:
# 3) Cài env — KHÔNG động vào numpy/transformers/torch (dùng bản Colab ship sẵn, đã tương thích).
#    Chỉ update bitsandbytes/peft/accelerate. (requirements.txt numpy>=1.26 nên KHÔNG hạ cấp numpy.)
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import numpy, torch, transformers, bitsandbytes as bnb
print('numpy', numpy.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| bnb', bnb.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x (session cu ban) -> Runtime > Disconnect and delete runtime, roi Run all lai')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
numpy 2.0.2 | torch 2.11.0+cu128 | transformers 5.12.1 | bnb 0.49.2 | CUDA True Tesla T4


In [4]:
# 4) SMOKE TEST 3 file trước (kiểm model + JSON parse; lần đầu tải model ~15GB)
import os, shutil
os.makedirs('smoke/input', exist_ok=True)
for n in [1, 5, 50]:
    shutil.copy(f'data/test/input/{n}.txt', f'smoke/input/{n}.txt')
!python src/pipeline.py --input smoke/input --output smoke/out --backend llm
print('----- smoke/out/5.json -----')
!cat smoke/out/5.json

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=103 brands=4150
[pipeline] backend=llm | 3 file | out=smoke/out
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0% 1/434 [00:12<1:27:51, 12.17s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 434/434 [02:00<00:00,  3.60it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[pipeline] xong.
----- 

In [5]:
# 5) Chạy full 100 file -> GHI VÀO GOOGLE DRIVE + --resume.
#    Colab ngắt/máy sleep -> chạy lại đúng cell này (sau cell 1-3): tự BỎ QUA file đã xong (đừng xoá output!).
#
# (Tùy chọn) đổi model few-shot 3B -> 7B (mạnh hơn, chậm hơn; đều ≤9B, KHÔNG train). Bỏ comment 2 dòng:
# import yaml; _c=yaml.safe_load(open('configs/config.yaml',encoding='utf-8')); _c['extract']['llm_model']='Qwen/Qwen2.5-7B-Instruct'
# yaml.safe_dump(_c, open('configs/config.yaml','w',encoding='utf-8'), allow_unicode=True, sort_keys=False)
OUT = '/content/drive/MyDrive/vmr_output'
# ⚠️ CHỈ bỏ comment khi ĐỔI MODEL / muốn chạy lại SẠCH (KHÔNG bỏ comment khi resume sau khi Colab ngắt):
# !rm -rf {OUT}
!python src/pipeline.py --input data/test/input --output {OUT} --backend llm --resume
!ls {OUT} | wc -l   # số file đã xong / 100

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=103 brands=4150
[pipeline] backend=llm | 100 file | out=/content/drive/MyDrive/vmr_output | resume
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0% 1/434 [00:02<19:45,  2.74s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 434/434 [00:27<00:00, 15.55it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(b

In [6]:
# 6) Validate + đóng gói (đọc output từ Drive) + tải về
OUT = '/content/drive/MyDrive/vmr_output'
!python scripts/package_submission.py --output {OUT} --input data/test/input --n 100
from google.colab import files; files.download('output.zip')

✅ Đã tạo output.zip (100 file hợp lệ).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# 7) Đo trên dev gold (copy output 2 cell này gửi lại để tinh chỉnh)
OUT = '/content/drive/MyDrive/vmr_output'
!python src/eval/official_scorer.py --pred {OUT} --gold data/dev/gold   # METRIC CHÍNH THỨC BTC
!python src/eval/scorer.py --pred {OUT} --gold data/dev/gold --mode overlap   # tham khảo (span-F1)
!python src/eval/eval_linking.py


=== METRIC CHÍNH THỨC (n=30 file) ===
  text_score       (0.3) = 0.1392
  assertions_score (0.3) = 0.2243
  candidates_score (0.4) = 0.1548
  ---------------------------------
  FINAL            = 0.1710

=== SPAN+TYPE (mode=overlap) ===
P=0.501  R=0.333  F1=0.400   (TP=254 nPred=507 nGold=763)

=== theo type (F1) ===
  CHẨN_ĐOÁN            F1=0.348  (tp=58 pred=123 gold=210)
  KẾT_QUẢ_XÉT_NGHIỆM   F1=0.484  (tp=31 pred=54 gold=74)
  THUỐC                F1=0.528  (tp=33 pred=48 gold=77)
  TRIỆU_CHỨNG          F1=0.453  (tp=105 pred=214 gold=250)
  TÊN_XÉT_NGHIỆM       F1=0.245  (tp=27 pred=68 gold=152)

Assertion exact-set acc = 0.883 (173/196)
Candidate hit (gold∈pred) = 0.681 (62/91)
[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=103 brands=4150

=== ICD-10 (CHẨN_ĐOÁN) ===
  mention: 210 | có mã gold: 194 | linker trả ≥1 mã: 201 (coverage=0.96)
  Jaccard exact       = 0.735   (= điểm candidates đóng góp)
  Jaccard nhóm3      = 0.871   (trần nế